# Big data, Assignment 6 | Matic Zadobovšek, Matija Bažec

## Preparing Kafka instance and Kafka Producer and Consumer

The Kafka cluster was started using the `docker-compose.yml` provided file. The Python code connects to Kafka through the exposed broker port:
`localhost:10000`

In [51]:
import pandas as pd

Each taxi trip is sent through Kafka as a message to the topic `yellow-taxi`.

In [52]:
BOOTSTRAP_SERVERS = 'localhost:10000'
TOPIC_NAME = 'yellow-taxi'

In [53]:
files = ['part.48.parquet', 'part.49.parquet', 'part.50.parquet']

for file in files:
    df = pd.read_parquet(file)
    # just checking all available columns
    print(df.columns)

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'pickup_date',
       'temp_max_c', 'temp_min_c', 'precipitation_mm', 'Borough_DO', 'Zone_DO',
       'service_zone_DO', 'Borough_PU', 'Zone_PU', 'service_zone_PU',
       'active_PU_events', 'active_DO_events', 'school_count',
       'attraction_count'],
      dtype='str')
Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'pi

We are only interested in certain columns for the streaming pipeline:
- pickup and dropoff timestamps
- pickup and dropoff locations
- borough and zone names
- numerical trip attributes
- additional augmented features from assignment 3

The data is sorted by `tpep_pickup_datetime` before being sent to Kafka, so the stream processing can process data in chronological order, just like if it were coming in real time from a data source.

In [54]:
# the main colmns we are interested in
cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "Borough_PU",
    "Zone_PU",
    "Borough_DO",
    "Zone_DO",
    "temp_max_c",
    "precipitation_mm",
    "active_PU_events",
    "active_DO_events",
]

df = pd.concat([pd.read_parquet(file) for file in files], ignore_index=True)

df = df[
    (df["tpep_pickup_datetime"] >= "2024-01-01") &
    (df["tpep_pickup_datetime"] < "2025-01-01")
].copy()
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)
# limit ourself for purpose of this assignment
# enough to simulate the stream
df_stream = df.head(100_000).copy()

In [55]:
df_stream.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,Borough_DO,Zone_DO,service_zone_DO,Borough_PU,Zone_PU,service_zone_PU,active_PU_events,active_DO_events,school_count,attraction_count
0,1,2024-01-01 00:00:00,2024-01-01 00:02:49,1.0,0.30,1.0,N,140,140,1,...,Manhattan,Lenox Hill East,Yellow Zone,Manhattan,Lenox Hill East,Yellow Zone,0,0,106,89
1,2,2024-01-01 00:00:02,2024-01-01 00:04:36,2.0,1.57,1.0,N,161,236,1,...,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Midtown Center,Yellow Zone,0,0,106,89
2,1,2024-01-01 00:00:03,2024-01-01 00:03:16,1.0,0.50,1.0,N,136,18,2,...,Bronx,Bedford Park,Boro Zone,Bronx,Kingsbridge Heights,Boro Zone,0,0,118,8
3,2,2024-01-01 00:00:04,2024-01-01 00:22:32,1.0,7.37,1.0,N,125,157,2,...,Queens,Maspeth,Boro Zone,Manhattan,Hudson Sq,Yellow Zone,0,0,80,12
4,2,2024-01-01 00:00:05,2024-01-01 00:26:44,2.0,18.85,2.0,N,132,41,1,...,Manhattan,Central Harlem,Boro Zone,Queens,JFK Airport,Airports,0,0,106,89


In [56]:
df_stream.tail()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,Borough_DO,Zone_DO,service_zone_DO,Borough_PU,Zone_PU,service_zone_PU,active_PU_events,active_DO_events,school_count,attraction_count
99995,1,2024-01-02 11:02:30,2024-01-02 11:07:03,1.0,0.50,1.0,N,162,233,1,...,Manhattan,UN/Turtle Bay South,Yellow Zone,Manhattan,Midtown East,Yellow Zone,0,0,106,89
99996,1,2024-01-02 11:02:33,2024-01-02 11:52:55,1.0,17.40,2.0,N,132,163,4,...,Manhattan,Midtown North,Yellow Zone,Queens,JFK Airport,Airports,0,0,106,89
99997,1,2024-01-02 11:02:35,2024-01-02 11:11:17,1.0,1.10,1.0,N,230,162,1,...,Manhattan,Midtown East,Yellow Zone,Manhattan,Times Sq/Theatre District,Yellow Zone,0,0,106,89
99998,2,2024-01-02 11:02:36,2024-01-02 11:36:11,2.0,16.34,2.0,N,132,233,1,...,Manhattan,UN/Turtle Bay South,Yellow Zone,Queens,JFK Airport,Airports,0,0,106,89
99999,2,2024-01-02 11:02:36,2024-01-02 11:11:42,1.0,1.43,1.0,N,142,239,1,...,Manhattan,Upper West Side South,Yellow Zone,Manhattan,Lincoln Square East,Yellow Zone,0,0,106,89


Kafka messages are sent as JSON strings, but the dataframe can contain values that are not directly serializable, so we have a helper function to convert missing values and also the datetime values to json-compatible formats.
Kafka `Producer` sends each row of the DataFrame to the Kafka topic.

In [151]:
import json
import socket
from confluent_kafka import Producer

conf = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "client.id": socket.gethostname(),
}
producer = Producer(conf)
# callback to handle printing reports
def report(err, msg):
    if err is not None:
        print(f"Message delivery failed: {err}")

# fix the nonserializable values in the record
# so everything can be serialized to json for Kafka
def json_fix(rec):
    for key, value in rec.items():
        if pd.isna(value):
            rec[key] = None
        elif hasattr(value, "isoformat"):
            rec[key] = value.isoformat()
    return rec

# processing row by row and sending to kafka
for count, row in df_stream.iterrows():
    rec = row[cols].to_dict()
    rec = json_fix(rec)
    # using pickup location ID key, so same pickup locations will be in the same partition
    producer.produce(TOPIC_NAME, key=str(rec["PULocationID"]), value=json.dumps(rec), callback=report)
    # serve reports
    producer.poll(0)
    # intermediate progress...
    if count % 10000 == 0:
        print(f"Produced {count} messages")

# gotta make sure that all msgs are sent
producer.flush()
print("Finished producing")

Produced 0 messages
Produced 10000 messages
Produced 20000 messages
Produced 30000 messages
Produced 40000 messages
Produced 50000 messages
Produced 60000 messages
Produced 70000 messages
Produced 80000 messages
Produced 90000 messages
Finished producing


Our producer has now sent taxi trip data to the kafka topic `yellow-taxi`. Now we can add a kafka consumer that will read from this topic and then do some stream processing.

The consumer now reads from the `yellow-taxi` topic and processes messages and stores them in a list for further analysis.
Manual offset commits were used (`enable.auto.commit=False`). Offsets are committed only after successful message processing (at-least once processing guarantee).

In [103]:
from confluent_kafka import Consumer, KafkaError, KafkaException
import time

# unique group id to avoid conflicts
grp =  f"yellow-taxi-consumer-{int(time.time())}"

# configuration
consumer_conf = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "group.id": grp,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
}

consumer = Consumer(consumer_conf)

# subscribe to the topic
consumer.subscribe([TOPIC_NAME])

# we want to gather messages until we have no more for a while
max_idle = 5
idle = 0
msgs = []

try:
    while True:
        msg = consumer.poll(timeout=2.0)
        if msg is None:
            idle += 1
            if idle >= max_idle:
                print("No more messages, exiting")
                break
            continue
        idle = 0
        # check for errors
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                raise KafkaException(msg.error())
        # decode the message value from json
        rec = json.loads(msg.value().decode("utf-8"))
        # storing
        msgs.append(rec)
        # commit the offset
        consumer.commit(msg, asynchronous=False)

finally:
    # free resources
    consumer.close()

No more messages, exiting


In [104]:
msgs_df = pd.DataFrame(msgs)

In [105]:
# basic stats
print(f"Total messages consumed: {len(msgs_df)}")

Total messages consumed: 100000


## Quix Streams for stream processing

In [117]:
from quixstreams import Application
from quixstreams.dataframe.windows import Count, Mean, Min, Max, Latest, Collect
import numpy as np
import math

In [118]:
df_stream.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'pickup_date',
       'temp_max_c', 'temp_min_c', 'precipitation_mm', 'Borough_DO', 'Zone_DO',
       'service_zone_DO', 'Borough_PU', 'Zone_PU', 'service_zone_PU',
       'active_PU_events', 'active_DO_events', 'school_count',
       'attraction_count'],
      dtype='str')

In [135]:
# combining pickup and dropoff counts
pickup_c = df_stream["PULocationID"].value_counts()
dropoff_c = df_stream["DOLocationID"].value_counts()

combined_location_counts = pickup_c.add(dropoff_c, fill_value=0).sort_values(ascending=False)
# getting top 10 location ids
top10_location_ids = set(combined_location_counts.head(10).index.astype(int))

print("Top 10 location IDs:")
print(combined_location_counts.head(10))
print(top10_location_ids)

Top 10 location IDs:
PULocationID
132    9498.0
161    6895.0
186    6166.0
79     5790.0
170    5728.0
48     5713.0
236    5636.0
237    5523.0
162    5446.0
230    5292.0
Name: count, dtype: float64
{161, 162, 132, 230, 170, 236, 237, 79, 48, 186}


In [136]:
STATS_TOPIC = "stats-calc"
# the attributes that we will be calculating rolling stats for
ATTRS = ["trip_distance", "fare_amount", "tip_amount"]
# each result is calculated over 1000 stream events per group
WINDOW_COUNT = 1000
# instance
app = Application(
    broker_address=BOOTSTRAP_SERVERS,
    consumer_group=f"yellow-taxi-stats-calc-{int(time.time())}",
    auto_offset_reset="earliest",
)
# the input and output topics
input_topic = app.topic(name=TOPIC_NAME, value_deserializer="json")
output_topic = app.topic(name=STATS_TOPIC, value_serializer="json")
sdf = app.dataframe(topic=input_topic)

[2026-05-16 12:18:52,433] [INFO] [quixstreams] : Creating a new topic "stats-calc" with a config: "{'num_partitions': 1, 'replication_factor': 1, 'extra_config': {}}"
[2026-05-16 12:18:53,434] [INFO] [quixstreams] : Topic "stats-calc" has been created


In [137]:
def to_float_or_none(x):
    if x is None:
        return None
    try:
        value = float(x)
        if math.isnan(value):
            return None
        return value
    except Exception:
        return None

def group_events(trip):
    trip_distance = to_float_or_none(trip.get("trip_distance"))
    fare_amount = to_float_or_none(trip.get("fare_amount"))
    tip_amount = to_float_or_none(trip.get("tip_amount"))

    # skip if missing values that we need
    if trip_distance is None or fare_amount is None or tip_amount is None:
        return []
    base = {
        "timestamp": trip.get("tpep_pickup_datetime"),
        "trip_distance": trip_distance,
        "fare_amount": fare_amount,
        "tip_amount": tip_amount,
    }
    events = []
    # borough level group
    borough = trip.get("Borough_PU")
    if borough is not None:
        events.append({
            "group_key": f"borough:{borough}",
            "group_type": "borough",
            "group": borough,
            "source": "pickup_borough",
            **base,
        })
    # top pickup location group
    pu_id = trip.get("PULocationID")
    if pu_id is not None:
        pu_id = int(pu_id)

        if pu_id in top10_location_ids:
            events.append({
                "group_key": f"location:{pu_id}",
                "group_type": "top_location",
                "group": pu_id,
                "source": "pickup",
                **base,
            })
    # top dropoff location group
    do_id = trip.get("DOLocationID")
    if do_id is not None:
        do_id = int(do_id)

        if do_id in top10_location_ids:
            events.append({
                "group_key": f"location:{do_id}",
                "group_type": "top_location",
                "group": do_id,
                "source": "dropoff",
                **base,
            })
    return events

In [139]:
# extra stats that we can't get from inbuilt functions
def extra_stats_from_events(events, attr):
    values = []
    # extract values
    for event in events:
        value = to_float_or_none(event.get(attr))
        if value is not None:
            values.append(value)
    # no values..
    if len(values) == 0:
        return {
            f"{attr}_median": None,
            f"{attr}_variance": None,
            f"{attr}_std": None,
        }
    arr = np.array(values, dtype=float)
    # doing calculations with numpy
    return {
        f"{attr}_median": float(np.median(arr)),
        f"{attr}_variance": float(np.var(arr, ddof=1)) if len(arr) > 1 else 0.0,
        f"{attr}_std": float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
    }

def finalize_stats(row):
    events = row["events"]
    result = {
        "window_start": row["start"],
        "window_end": row["end"],
        "window_count": WINDOW_COUNT,
        "group_key": row["group_key"],
        "group_type": row["group_type"],
        "group": row["group"],
        "count": row["count"],
    }
    for attr in ATTRS:
        # built-in aggregations
        result[f"{attr}_mean"] = row[f"{attr}_mean"]
        result[f"{attr}_min"] = row[f"{attr}_min"]
        result[f"{attr}_max"] = row[f"{attr}_max"]
        # extra calculations
        result.update(extra_stats_from_events(events, attr))
    return result

In [ ]:
stats_sdf = (
    sdf
    # each trip is converted into one of the group events (orough group, top pickup location group, top dropoff location group)
    .apply(group_events, expand=True)
    # all events for the same group are processed together
    .group_by("group_key")
    # collect 1000 events and calculate stats on them for each group
    .tumbling_count_window(WINDOW_COUNT)
    # aggregations we want to calculate for each group in the window
    .agg(
        count=Count(),
        group_key=Latest("group_key"),
        group_type=Latest("group_type"),
        group=Latest("group"),
        # for trip distance
        trip_distance_mean=Mean("trip_distance"),
        trip_distance_min=Min("trip_distance"),
        trip_distance_max=Max("trip_distance"),
        # for fare amount
        fare_amount_mean=Mean("fare_amount"),
        fare_amount_min=Min("fare_amount"),
        fare_amount_max=Max("fare_amount"),
        # for tip amount
        tip_amount_mean=Mean("tip_amount"),
        tip_amount_min=Min("tip_amount"),
        tip_amount_max=Max("tip_amount"),
        # used for extra calculations that we can't do with built-in aggs
        # median, variance and standard deviation for each attribute
        events=Collect(),
    )
    # emit the results
    .final()
    .apply(finalize_stats)
)

[2026-05-16 12:19:40,096] [INFO] [quixstreams] : Creating a new topic "changelog__yellow-taxi-stats-calc-1778926732--yellow-taxi--groupby--group_key--tumbling_count_window" with a config: "{'num_partitions': 1, 'replication_factor': 1, 'extra_config': {'retention.bytes': '-1', 'retention.ms': '3600000', 'cleanup.policy': 'compact'}}"
[2026-05-16 12:19:41,097] [INFO] [quixstreams] : Topic "changelog__yellow-taxi-stats-calc-1778926732--yellow-taxi--groupby--group_key--tumbling_count_window" has been created


In [143]:
stats_sdf = stats_sdf.update(lambda row: print(row))
stats_sdf.to_topic(output_topic)
print(f"Starting app.")
print(f"Input topic:  {TOPIC_NAME}")
print(f"Output topic: {STATS_TOPIC}")
print("Stop the cell manually...")
app.run()

[2026-05-16 12:21:17,843] [INFO] [quixstreams] : Starting the Application with the config: broker_address="{'bootstrap.servers': 'localhost:10000'}" consumer_group="yellow-taxi-stats-calc-1778926732" auto_offset_reset="earliest" commit_interval=5.0s commit_every=0 processing_guarantee="at-least-once"
[2026-05-16 12:21:17,843] [INFO] [quixstreams] : Initializing state directory at "/home/7matic/Documents/Big data/bd_project/Assignment6/state/yellow-taxi-stats-calc-1778926732"
[2026-05-16 12:21:17,844] [INFO] [quixstreams] : The application started and is now processing incoming messages


Starting app.
Input topic:  yellow-taxi
Output topic: stats-calc
Stop the cell manually...
{'window_start': 1778926827835, 'window_end': 1778926828263, 'window_count': 1000, 'group_key': 'borough:Manhattan', 'group_type': 'borough', 'group': 'Manhattan', 'count': 1000, 'trip_distance_mean': 2.420270000000001, 'trip_distance_min': 0.0, 'trip_distance_max': 18.23, 'trip_distance_median': 1.79, 'trip_distance_variance': 4.590002229329329, 'trip_distance_std': 2.142429048843702, 'fare_amount_mean': 15.897349999999989, 'fare_amount_min': -10.0, 'fare_amount_max': 94.7, 'fare_amount_median': 12.8, 'fare_amount_variance': 116.25885333083085, 'fare_amount_std': 10.78233988199365, 'tip_amount_mean': 3.0152300000000007, 'tip_amount_min': 0.0, 'tip_amount_max': 90.0, 'tip_amount_median': 2.72, 'tip_amount_variance': 15.498923070170171, 'tip_amount_std': 3.936867164404988}
{'window_start': 1778926827835, 'window_end': 1778926828263, 'window_count': 1000, 'group_key': 'borough:Manhattan', 'group_ty

[2026-05-16 12:23:03,666] [INFO] [quixstreams] : Stopping the application


[]

In [144]:
# configuration
consumer_conf = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "group.id": f"stats-check-{int(time.time())}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
}

consumer = Consumer(consumer_conf)

# subscribe to the topic
consumer.subscribe(["stats-calc"])

# we want to gather messages until we have no more for a while
max_idle = 5
idle = 0
msgs = []

try:
    while True:
        msg = consumer.poll(timeout=2.0)
        if msg is None:
            idle += 1
            if idle >= max_idle:
                print("No more messages, exiting")
                break
            continue
        idle = 0
        # check for errors
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                raise KafkaException(msg.error())
        # decode the message value from json
        rec = json.loads(msg.value().decode("utf-8"))
        # storing
        msgs.append(rec)
        # commit the offset
        consumer.commit(msg, asynchronous=False)

finally:
    # free resources
    consumer.close()

stats_df = pd.DataFrame(msgs)
print("Stats messages consumed:", len(stats_df))

No more messages, exiting
Stats messages consumed: 308


In [145]:
stats_df.sample(20)

,window_start,window_end,window_count,group_key,group_type,group,count,trip_distance_mean,trip_distance_min,trip_distance_max,...,fare_amount_max,fare_amount_median,fare_amount_variance,fare_amount_std,tip_amount_mean,tip_amount_min,tip_amount_max,tip_amount_median,tip_amount_variance,tip_amount_std
60,1778926837309,1778926837739,1000,borough:Manhattan,borough,Manhattan,1000,2.98353,0.0,27.76,...,226.49,15.850,299.992069,17.320279,2.15938,0.0,26.09,1.000,9.579392,3.095059
219,1778926856371,1778926856878,1000,borough:Manhattan,borough,Manhattan,1000,2.21987,0.0,27.30,...,95.00,11.400,105.365217,10.264756,2.61337,0.0,29.00,2.580,6.611770,2.571336
109,1778926840466,1778926843802,1000,location:132,top_location,132,1000,15.00511,0.0,60.93,...,266.00,70.000,875.316443,29.585747,8.50660,0.0,56.20,8.495,64.218795,8.013663
89,1778926840311,1778926840866,1000,borough:Manhattan,borough,Manhattan,1000,4.53614,0.0,26.41,...,136.00,12.800,549.191678,23.434839,3.05593,0.0,40.55,1.000,25.517653,5.051500
3,1778926828263,1778926828688,1000,borough:Manhattan,borough,Manhattan,1000,2.51290,0.0,27.08,...,150.00,13.500,186.023451,13.639041,3.01365,0.0,30.00,2.720,10.623227,3.259329
27,1778926827845,1778926833176,1000,location:170,top_location,170,1000,2.10416,0.0,26.71,...,125.50,14.900,145.388649,12.057722,2.60352,0.0,81.11,2.440,14.020434,3.744387
293,1778926858478,1778926864381,1000,location:230,top_location,230,1000,4.10011,0.0,25.90,...,124.80,12.100,474.110730,21.774084,2.96546,0.0,29.31,2.000,18.139198,4.259014
237,1778926853079,1778926858478,1000,location:230,top_location,230,1000,3.17266,0.0,30.09,...,143.00,13.500,358.534786,18.935015,2.43403,0.0,39.99,0.020,16.248038,4.030885
259,1778926858953,1778926860448,1000,location:132,top_location,132,1000,14.53734,0.0,77.26,...,442.60,70.000,1000.257139,31.626842,8.15038,0.0,48.08,8.385,55.919755,7.477951
244,1778926858160,1778926859061,1000,borough:Queens,borough,Queens,1000,13.74895,0.0,77.26,...,442.60,56.900,1007.242992,31.737092,8.60323,0.0,50.00,9.530,46.028771,6.784451


In [146]:
# show one borough example
borough_example = (
    stats_df[stats_df["group_type"] == "borough"]
    .sort_values(["group", "window_start"])
    .head(1)
)

borough_example.T

,70
window_start,1778926827855
window_end,1778926839107
window_count,1000
group_key,borough:Brooklyn
group_type,borough
group,Brooklyn
count,1000
trip_distance_mean,4.41134
trip_distance_min,0.0
trip_distance_max,23.36


In [147]:
# show one top location example
location_example = (
    stats_df[stats_df["group_type"] == "top_location"]
    .sort_values(["group", "window_start"])
    .head(1)
)

location_example.T

,44
window_start,1778926827845
window_end,1778926835357
window_count,1000
group_key,location:48
group_type,top_location
group,48
count,1000
trip_distance_mean,2.64707
trip_distance_min,0.0
trip_distance_max,24.83


In [149]:
cols_to_show = [
    "group_key",
    "group_type",
    "group",
    "count",
    "trip_distance_mean",
    "trip_distance_median",
    "trip_distance_std",
    "fare_amount_mean",
    "fare_amount_median",
    "fare_amount_std",
    "tip_amount_mean",
    "tip_amount_median",
    "tip_amount_std",
]

example_results = pd.concat([borough_example, location_example])[cols_to_show]
example_results

,group_key,group_type,group,count,trip_distance_mean,trip_distance_median,trip_distance_std,fare_amount_mean,fare_amount_median,fare_amount_std,tip_amount_mean,tip_amount_median,tip_amount_std
70,borough:Brooklyn,borough,Brooklyn,1000,4.41134,3.540,4.040028,30.11680,28.20,19.708894,2.22341,0.00,4.061557
44,location:48,top_location,48,1000,2.64707,2.025,2.598697,21.54176,18.86,17.609094,3.16235,2.86,3.511875


group=132 is the JFK airport, so it makes sense that the trip distance and fare amount are higher, because the distance is much longer. 

In [150]:
examples = pd.concat([
    stats_df[
        (stats_df["group_type"] == "borough") &
        (stats_df["group"] == "Manhattan")
    ].head(1),

    stats_df[
        (stats_df["group_type"] == "top_location") &
        (stats_df["group"] == 132)
    ].head(1)
])

examples[cols_to_show]

,group_key,group_type,group,count,trip_distance_mean,trip_distance_median,trip_distance_std,fare_amount_mean,fare_amount_median,fare_amount_std,tip_amount_mean,tip_amount_median,tip_amount_std
0,borough:Manhattan,borough,Manhattan,1000,2.42027,1.790,2.142429,15.89735,12.8,10.782340,3.01523,2.72,3.936867
86,location:132,top_location,132,1000,14.24410,16.765,6.923688,56.11241,70.0,28.914409,7.91421,7.42,7.787300


For the streaming clustering algorithm we decided to implement the **BIRCH** algorithm, which is a hierarchical clustering algorithm. Each of the trip data becomes a point in the feature space defined by the selected features (trip distance, fare amount, tip amount, passenger count, for example), and the BIRCH algorithm is able to group similar points together into clusters. Afterwards we can calculate summary statistics for each cluster, to see what kind of trips are in each cluster.

BIRCH is organized as a CF-tree. When a new taxi trip is added, we try to find the closest subcsluter in the tree. If it can be added to that subcluster, then we add it there and update the summary. If it does not fit, then we need to create a new subcluster. And th last scenario is if the number of subclusters already exceeds the maximum number, then the tree needs to be splitted.

In [ ]:
# for the clustering part
CLUSTER_TOPIC = "taxi-birch"
# features that will define the feature space
FEATURES = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "passenger_count",
]
# how many records to use initially before creating the first model
INIT_SIZE = 2000  
# how many records process in each batch
BATCH_SIZE = 1000 
# smaller = more clusters, larger = fewer clusters
BIRCH_THRESHOLD = 0.7

In [ ]:
# same configuration as before for the consumer and producer...
consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "group.id": f"taxi-birch-{int(time.time())}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})
producer = Producer({
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "client.id": socket.gethostname(),
})
consumer.subscribe([TOPIC_NAME])

In [ ]:
# some additional stuff that we need for clustering
from sklearn.cluster import Birch
from sklearn.preprocessing import StandardScaler
from collections import Counter
import pandas as pd
import numpy as np
import json
import time
import socket

In [ ]:
scaler = StandardScaler()
# initializing the BIRCH model
birch = Birch(
    threshold=BIRCH_THRESHOLD,
    branching_factor=50,
    n_clusters=None,
)
model_ready = False
init_records = []
batch_records = []
clustered_rows = []
cluster_counts = Counter()

In [ ]:
# for each record we get the needed features and convert them into a list of values
def extract_features(record):
    values = []
    for feature in FEATURES:
        value = record.get(feature)
        if value is None:
            return None
        try:
            value = float(value)
        except Exception:
            return None
        if not np.isfinite(value):
            return None
        values.append(value)
    trip_distance, fare_amount, tip_amount, passenger_count = values
    return values

In [ ]:
def produce_cluster_result(record, cluster_id):
    result = {
        "timestamp": record.get("tpep_pickup_datetime"),
        "PULocationID": record.get("PULocationID"),
        "DOLocationID": record.get("DOLocationID"),
        "Borough_PU": record.get("Borough_PU"),
        "Borough_DO": record.get("Borough_DO"),
        "cluster": int(cluster_id),
    }
    for feature in FEATURES:
        result[feature] = float(record[feature])
    # send the result to kafka
    producer.produce(
        CLUSTER_TOPIC,
        key=str(cluster_id),
        value=json.dumps(result),
    )
    producer.poll(0)
    clustered_rows.append(result)
    cluster_counts[int(cluster_id)] += 1

In [ ]:
def fit_initial_model(records):
    global model_ready
    valid_records = []
    feature_rows = []
    for record in records:
        features = extract_features(record)
        if features is not None:
            valid_records.append(record)
            feature_rows.append(features)
    if len(feature_rows) == 0:
        return
    X = np.array(feature_rows, dtype=float)
    # fit only on the initial batch
    # keep feature space stable for the stream
    scaler.fit(X)
    X_scaled = scaler.transform(X)
    # Initialize birch incremenetally
    birch.partial_fit(X_scaled)
    # assign initial records to BIRCH subclusters
    labels = birch.predict(X_scaled)
    for record, label in zip(valid_records, labels):
        produce_cluster_result(record, label)
    model_ready = True

In [ ]:
def process_batch(records):
    valid_records = []
    feature_rows = []
    for record in records:
        features = extract_features(record)
        if features is not None:
            valid_records.append(record)
            feature_rows.append(features)
    if len(feature_rows) == 0:
        return
    X = np.array(feature_rows, dtype=float)
    X_scaled = scaler.transform(X)
    # predict current labels
    labels = birch.predict(X_scaled)
    for record, label in zip(valid_records, labels):
        produce_cluster_result(record, label)
    # update the model with the new batch
    birch.partial_fit(X_scaled)

In [ ]:
max_idle = 5
idle = 0
messages_seen = 0

try:
    while True:
        msg = consumer.poll(timeout=2.0)
        if msg is None:
            idle += 1
            if idle >= max_idle:
                print("No more input messages. Stopping BIRCH clustering consumer.")
                break
            continue
        idle = 0
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                raise KafkaException(msg.error())
        record = json.loads(msg.value().decode("utf-8"))
        messages_seen += 1
        if not model_ready:
            init_records.append(record)
            if len(init_records) >= INIT_SIZE:
                fit_initial_model(init_records)
                init_records = []
        else:
            batch_records.append(record)
            if len(batch_records) >= BATCH_SIZE:
                process_batch(batch_records)
                batch_records = []
        consumer.commit(msg, asynchronous=False)
finally:
    if not model_ready and len(init_records) > 0:
        fit_initial_model(init_records)
    if model_ready and len(batch_records) > 0:
        process_batch(batch_records)
    producer.flush()
    consumer.close()

No more input messages. Stopping BIRCH clustering consumer.


In [164]:
cluster_df = pd.DataFrame(clustered_rows)

print(f"Input messages seen: {messages_seen}")
print(f"Clustered trips: {len(cluster_df)}")
print(f"Number of BIRCH clusters {cluster_df['cluster'].nunique() if len(cluster_df) else 0}")
print("Cluster counts:")
print(cluster_counts)

Input messages seen: 100000
Clustered trips: 88516
Number of BIRCH clusters 679
Cluster counts:
Counter({183: 1567, 157: 1547, 0: 1528, 154: 1336, 162: 1323, 190: 1312, 110: 1084, 107: 1049, 47: 980, 160: 974, 186: 929, 146: 869, 145: 851, 159: 850, 133: 848, 148: 838, 161: 831, 187: 831, 113: 819, 189: 791, 153: 745, 165: 744, 151: 741, 71: 733, 164: 729, 169: 721, 136: 714, 142: 695, 139: 693, 185: 688, 193: 682, 192: 681, 127: 676, 191: 670, 122: 667, 73: 657, 80: 650, 130: 649, 155: 638, 120: 633, 135: 618, 108: 608, 115: 573, 150: 573, 50: 569, 163: 568, 6: 560, 141: 558, 147: 557, 114: 555, 116: 539, 188: 537, 81: 504, 143: 494, 138: 492, 166: 492, 168: 489, 194: 486, 74: 482, 92: 477, 117: 457, 158: 455, 118: 453, 172: 452, 98: 450, 112: 445, 89: 437, 75: 430, 84: 429, 137: 425, 149: 421, 111: 412, 66: 404, 128: 402, 152: 398, 56: 397, 134: 392, 63: 391, 105: 390, 95: 383, 126: 381, 26: 379, 132: 376, 41: 375, 156: 366, 176: 360, 16: 353, 68: 353, 45: 352, 125: 352, 181: 347, 18

In [ ]:
cluster_profile = (
    cluster_df
    # grouping by the cluster
    .groupby("cluster")
    # calculating summary statistics for each cluster
    .agg(
        count=("cluster", "size"),
        mean_distance=("trip_distance", "mean"),
        median_distance=("trip_distance", "median"),
        mean_fare=("fare_amount", "mean"),
        median_fare=("fare_amount", "median"),
        mean_tip=("tip_amount", "mean"),
        median_tip=("tip_amount", "median"),
        mean_passengers=("passenger_count", "mean"),
    )
    .sort_values("count", ascending=False)
)

cluster_profile.head(20)

,count,mean_distance,median_distance,mean_fare,median_fare,mean_tip,median_tip,mean_passengers
cluster,,,,,,,,
183,1567,1.649955,1.47,10.364710,10.0,2.736063,2.800,1.008934
157,1547,1.579515,1.44,11.042275,10.7,2.291907,2.520,1.265676
0,1528,5.576243,5.45,26.793757,26.8,0.341217,0.000,0.992147
154,1336,1.643323,1.50,11.101272,10.7,2.462882,2.650,1.238772
162,1323,1.575616,1.40,10.743462,10.0,2.670454,2.720,1.403628
190,1312,1.247416,0.96,8.381250,7.9,1.347706,1.000,1.127287
110,1084,1.836697,1.50,11.882749,10.7,2.974705,2.860,1.493542
107,1049,1.952545,1.54,12.306864,10.7,3.458713,3.050,1.042898
47,980,2.020224,1.28,13.067041,10.0,2.671082,2.720,1.223469


In [ ]:
consumer = Consumer({
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "group.id": f"birch-cluster-check-{int(time.time())}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})

consumer.subscribe([CLUSTER_TOPIC])

msgs = []
idle = 0
max_idle = 5

try:
    while True:
        msg = consumer.poll(timeout=2.0)
        if msg is None:
            idle += 1
            if idle >= max_idle:
                break
            continue
        idle = 0
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            else:
                raise KafkaException(msg.error())
        msgs.append(json.loads(msg.value().decode("utf-8")))
finally:
    consumer.close()

In [167]:
birch_kafka_df = pd.DataFrame(msgs)

print("Cluster messages consumed:", len(birch_kafka_df))
birch_kafka_df.head()

Cluster messages consumed: 88516


,timestamp,PULocationID,DOLocationID,Borough_PU,Borough_DO,cluster,trip_distance,fare_amount,tip_amount,passenger_count
0,2024-01-01T00:00:00,140,140,Manhattan,Manhattan,47,0.30,4.4,1.85,1.0
1,2024-01-01T00:00:02,161,236,Manhattan,Manhattan,50,1.57,8.6,2.72,2.0
2,2024-01-01T00:00:03,136,18,Bronx,Bronx,47,0.50,5.1,0.00,1.0
3,2024-01-01T00:00:04,125,157,Manhattan,Queens,0,7.37,33.8,0.00,1.0
4,2024-01-01T00:00:05,132,41,Queens,Manhattan,60,18.85,70.0,15.69,2.0
